In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /home/tuptc/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /home/tuptc/big-data-postgraduate
Salidas: /home/tuptc/big-data-postgraduate/kit/salidas/soluciones/PD03


In [2]:
#1. Dimensión Territorial
# Extracción de codigo_municipio
divipola["codigo_municipio"] = divipola["Código Municipio"].str.strip()

# Validación del formato (5 digitos)
es_valido = divipola["codigo_municipio"].str.fullmatch(r"[0-9]{5}")
assert es_valido.all(), "¡Ojo: hay códigos municipales de DIVIPOLA con formato inválido!"

# Validar unicidad en codigos y sin nulos
assert divipola["codigo_municipio"].is_unique, "¡Ojo: hay códigos municipales repetidos en DIVIPOLA!"
assert divipola["codigo_municipio"].str.strip().ne("").all(), "¡Ojo: Hay códigos municipales vacíos en DIVIPOLA!"

# Conteo de distribución por tipo de entidad territorial
conteo_tipos = divipola["Tipo: Municipio / Isla / Área no municipalizada"].value_counts()
print("--- DISTRIBUCIÓN POR TIPO EN DIVIPOLA ---")
display(conteo_tipos)

# Construcción de la tabla 'dimension'
dimension = divipola[["codigo_municipio","Nombre Departamento","Nombre Municipio","Tipo: Municipio / Isla / Área no municipalizada"]].copy()

print(f"\nDimensión territorial construida con {len(dimension):,} elementos.")
display(dimension.head(7))

--- DISTRIBUCIÓN POR TIPO EN DIVIPOLA ---


Tipo: Municipio / Isla / Área no municipalizada
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64


Dimensión territorial construida con 1,122 elementos.


,codigo_municipio,Nombre Departamento,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada
0,05001,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05021,ANTIOQUIA,ALEJANDRÍA,Municipio
4,05030,ANTIOQUIA,AMAGÁ,Municipio
5,05031,ANTIOQUIA,AMALFI,Municipio
6,05034,ANTIOQUIA,ANDES,Municipio


In [3]:
#2. Códigos EVA sin correspondencia

# Selección de año 
eva["anio"] = pd.to_numeric(eva["Año"].str.strip(), errors="coerce").astype("Int64") #Formato entero

# Selección del código municipal
codigo_raw = eva["Código Dane municipio"].str.strip()

# Vaerificación de formato numerico
es_numerico = codigo_raw.str.fullmatch(r"[0-9]+")

# Completar con ceros a izquierda los de formato valido
eva["codigo_municipio"] = codigo_raw.where(~es_numerico, codigo_raw.str.zfill(5))

# Generacion de combinaciones únicas codigo_municipio-anio en EVA
comb_eva = eva[["codigo_municipio", "anio"]].drop_duplicates().copy()

# Anti-join (Verificación de codigos en EVA que no existen en DIVIPOLA) 
cruce = comb_eva.merge(dimension[["codigo_municipio"]],on="codigo_municipio",how="left",indicator=True,validate="many_to_one")

# Filtro de combinaciones sin correspondencia en DIVIPOLA
sin_correspondencia = cruce.query('_merge == "left_only"').drop(columns=["_merge"]).reset_index(drop=True)

# Display
print(f"Total de combinaciones únicas (código, año) en EVA: {len(comb_eva):,}")
print(f"Combinaciones sin correspondencia en DIVIPOLA: {len(sin_correspondencia):,}")

print("\n--- Listado de sin_correspondencia ---")
display(sin_correspondencia)

Total de combinaciones únicas (código, año) en EVA: 7,691
Combinaciones sin correspondencia en DIVIPOLA: 0

--- Listado de sin_correspondencia ---


,codigo_municipio,anio


In [4]:
# 3.Selección para el indicador
# Conversión de vaariables a tipo numerico
eva["area_ha"] = pd.to_numeric(
    eva["Área cosechada"].str.strip().str.replace(",", ".", regex=False),errors="coerce")
eva["produccion_t"] = pd.to_numeric(
    eva["Producción"].str.strip().str.replace(",", ".", regex=False),errors="coerce")

# Filtro para el subconjunto "Papa 2024"
papa_2024 = eva.loc[eva["Cultivo"].str.strip().eq("Papa") & eva["anio"].eq(2024)].copy()

# Selección según criterios de area cosechada y producción en "Apta"
mask_apta = papa_2024["area_ha"].gt(0) & papa_2024["produccion_t"].ge(0)
apta = papa_2024.loc[mask_apta].copy()

# Selección de aquellos que quedaron fuera de "Apta"
excluidas = papa_2024.loc[~mask_apta].copy()

# Resumen
total_papa_2024 = len(papa_2024)
total_aptas = len(apta)
total_excluidas = len(excluidas)

# Comprobación de integridad con assert
assert total_papa_2024 == total_aptas + total_excluidas, "¡Error de comprobación!"
#Display
print(f"Total registros Papa 2024:  {total_papa_2024:,}")
print(f"Registros en 'apta':        {total_aptas:,}")
print(f"Registros excluidos:        {total_excluidas:,}")

#En caso de existir excluidos (no validos)
if total_excluidas > 0:
    print("\n--- Algunos Registros excluidos ---")
    display(excluidas[["codigo_municipio", "Cultivo", "area_ha", "produccion_t"]].head(5))

Total registros Papa 2024:  831
Registros en 'apta':        779
Registros excluidos:        52

--- Algunos Registros excluidos ---


,codigo_municipio,Cultivo,area_ha,produccion_t
2602,05093,Papa,0.0,0.0
23281,15047,Papa,0.0,0.0
23834,15092,Papa,0.0,0.0
24094,15104,Papa,0.0,0.0
24367,15109,Papa,0.0,0.0


In [5]:
#4. Rendimiento agregado
# Agrupación de la tabla 'apta' calculando conteo (contribución por municipio), suma de producción y del total de hectareas
rendimiento = apta.groupby(
    ["codigo_municipio", "Cultivo", "Estado físico del cultivo", "anio"],as_index=False,dropna=False).agg(registros=("area_ha", "size"),produccion_t_sum=("produccion_t", "sum"),area_ha_sum=("area_ha", "sum"))

# Calculo del rendimiento agregado (t/ha) (No puede hacerse sin el filtro del punto previo)
rendimiento["rendimiento_t_ha"] = rendimiento["produccion_t_sum"] / rendimiento["area_ha_sum"]

# Validaciones con assert (control e integridad)
assert (rendimiento["area_ha_sum"] > 0).all(), "¡Ojo: Hay valores para rendimiento inconsistentes!" #Negativos o divisiones por cero
assert rendimiento["registros"].sum() == len(apta), "¡Ojo: La suma de registros agrupados no coincide con los registros de apta)!"

print(f" Registros agrupados en 'apta': {len(apta):,}")
print(f"Suma de elementos en la columna 'registros' en rendimiento: {rendimiento['registros'].sum():,}")
print(f"Grupos resultantes: {len(rendimiento):,}")

print("\n--- Algunos registros de 'Rendimiento' ---")
display(rendimiento.head(5))

 Registros agrupados en 'apta': 779
Suma de elementos en la columna 'registros' en rendimiento: 779
Grupos resultantes: 292

--- Algunos registros de 'Rendimiento' ---


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t_sum,area_ha_sum,rendimiento_t_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459


In [6]:
#5. Unión con nombres y clasificación

# Unir rendimiento con dimension con left merge
rendimiento_municipal = rendimiento.merge(dimension, on="codigo_municipio",how="left",indicator=True,validate="many_to_one")

# Verificación de consistencia en tuplas
assert len(rendimiento_municipal) == len(rendimiento), "¡Ojo: La cantidad de filas cambió durante la ejecución del merge!"

# Verificación de códigos sin correspondencia territorial
sin_nombre = rendimiento_municipal.query('_merge == "left_only"')
cant_sin_nombre = len(sin_nombre)

print(f"Cantidad de Tuplas antes del merge: {len(rendimiento):,}")
print(f"Cantidad de Tuplas después del merge: {len(rendimiento_municipal):,}")
print(f"Registros sin nombre territorial: {cant_sin_nombre:,}")

if cant_sin_nombre > 0:
    print("\n--- Lista de códigos sin nombre en DIVIPOLA ---")
    display(sin_nombre[["codigo_municipio", "Cultivo", "rendimiento_t_ha"]])
else:
    print("\n¡Todos los códigos tienen un nombre en DIVIPOLA!")

# Eliminar columna producto del merge
rendimiento_municipal = rendimiento_municipal.drop(columns=["_merge"])
display(rendimiento_municipal.head(5))

Cantidad de Tuplas antes del merge: 292
Cantidad de Tuplas después del merge: 292
Registros sin nombre territorial: 0

¡Todos los códigos tienen un nombre en DIVIPOLA!


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t_sum,area_ha_sum,rendimiento_t_ha,Nombre Departamento,Nombre Municipio,Tipo: Municipio / Isla / Área no municipalizada
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000,ANTIOQUIA,MEDELLÍN,Municipio
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000,ANTIOQUIA,ABEJORRAL,Municipio
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000,ANTIOQUIA,ABRIAQUÍ,Municipio
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353,ANTIOQUIA,BARBOSA,Municipio
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459,ANTIOQUIA,BELMIRA,Municipio


In [7]:
# 6. Exportación e interpretación
# Definición de Rutas de Salida
ruta_rendimiento = OUT / "rendimiento_municipal.csv"
ruta_no_corr = OUT / "codigos_sin_correspondencia.csv"

# Exportación de dataframes a CSV (no Index)
rendimiento_municipal.to_csv(ruta_rendimiento, index=False, encoding="utf-8-sig")
sin_correspondencia.to_csv(ruta_no_corr, index=False, encoding="utf-8-sig")

# Validar existencia y estado de archivos exportados
assert ruta_rendimiento.is_file(), f"No se encontró el archivo {ruta_rendimiento}"
assert ruta_no_corr.is_file(), f"No se encontró el archivo {ruta_no_corr}"

print("Archivos exportados como sigue:")
print(f" Rendimiento municipal: {ruta_rendimiento}")
print(f" Códigos sin correspondencia: {ruta_no_corr} (Registros: {len(sin_correspondencia)})")

Archivos exportados como sigue:
 Rendimiento municipal: /home/tuptc/big-data-postgraduate/kit/salidas/soluciones/PD03/rendimiento_municipal.csv
 Códigos sin correspondencia: /home/tuptc/big-data-postgraduate/kit/salidas/soluciones/PD03/codigos_sin_correspondencia.csv (Registros: 0)


**Interpretación de resultados:**

1. **Razón de sumas vs. Promedio de rendimientos:**
La necesidad de utilizar la razón de "Suma de Producción / Suma de Área" ocurre debido al sesgo que existiría si diera un mismo peso a un predio pequeño que a uno grande. Un simple promedio de rendimientos en este caso distorsiona la productividad real de la zona

2. **Sobre la limitación temporal de DIVIPOLA....:**
Al utilizar DIVIPOLA de 2024 para datos históricos se asume que la distribución de municipios no ha cambiado. desconociendo eventos como creación de municipios o redistribucion de limites territoriales por parte del DANE